# DPO Tuning: Qwen3.5-4B — unsloth `PatchDPOTrainer` + LoRA (answer-last)

## 1. Install (Qwen3.5 + unsloth + trl 0.24) — 실행 후 런타임 재시작

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        --refresh-package unsloth_zoo --reinstall-package unsloth_zoo \
        --refresh-package unsloth    --reinstall-package unsloth \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.24.0
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
!apt-get update -qq && apt-get install -y -qq libz3-dev

## 2. Mount Drive & HuggingFace login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN     = userdata.get('HF_TOKEN')
HF_REPO_IN   = 'minsu0567/IAD-X1-SFT-answer-last'
HF_REPO_OUT  = 'minsu0567/IAD-X1-DPO-answer_last'
ADAPTER_REPO = 'minsu0567/IAD-X1-DPO-answer-last-adapter'
login(token=HF_TOKEN)
print('HF login OK.')
print('IN :', HF_REPO_IN)
print('OUT:', HF_REPO_OUT)

## 3. GPU check

In [ ]:
!nvidia-smi

## 4. Paths

In [ ]:
import os, sys, json

DRIVE_ROOT = '/content/drive/MyDrive'
GRPO_DIR   = f'{DRIVE_ROOT}/GRPO_dataset3'
DPO_SRC    = f'{DRIVE_ROOT}/IAD-X1/dpo_src'
DPO_JSON   = f'{DRIVE_ROOT}/dpo_hard_samples_answer_last.json'
OUTPUT_DIR = '/content/DPO_output/Qwen3_5_4B'

assert os.path.isdir(GRPO_DIR),  f'Missing dir: {GRPO_DIR}'
assert os.path.isfile(DPO_JSON), f'Missing: {DPO_JSON}'
assert os.path.isfile(f'{DPO_SRC}/dpo_pipeline.py'), f'Missing: {DPO_SRC}/dpo_pipeline.py'

if DPO_SRC not in sys.path:
    sys.path.insert(0, DPO_SRC)

with open(DPO_JSON, encoding='utf-8') as f:
    _raw = json.load(f)
assert _raw, 'DPO dataset is empty.'
_need = {'images', 'messages', 'chosen', 'rejected'}
_bad = [i for i, s in enumerate(_raw) if not _need <= set(s)]
assert not _bad, f'Unexpected schema at indices {_bad[:5]}'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('DPO json :', DPO_JSON, f'({len(_raw)} preference pairs)')
print('Output   :', OUTPUT_DIR)

## 5. Config

In [ ]:
LORA_R       = 64
LORA_ALPHA   = 64
LORA_DROPOUT = 0
RANDOM_STATE = 3407

LEARNING_RATE = 1e-5
ADAM_BETA1    = 0.9
ADAM_BETA2    = 0.99
WEIGHT_DECAY  = 0.0
WARMUP_RATIO  = 0.1
LR_SCHEDULER  = 'cosine'
MAX_GRAD_NORM = 1.0
OPTIM         = 'adamw_8bit'

BETA          = 0.1
PER_DEVICE_BS = 1
GRAD_ACCUM    = 4

MAX_STEPS  = 247
SAVE_STEPS = MAX_STEPS

MAX_PROMPT_LEN     = 8192
MAX_COMPLETION_LEN = 640
MAX_LEN            = MAX_PROMPT_LEN + MAX_COMPLETION_LEN
IMG_RESOLUTION     = 512
MAX_SEQ_LENGTH     = 16384

MAX_SAMPLES = None
SEED        = 42

print(f'LoRA r={LORA_R} alpha={LORA_ALPHA} | lr={LEARNING_RATE} | beta={BETA}')
print(f'batch {PER_DEVICE_BS} x accum {GRAD_ACCUM} = effective {PER_DEVICE_BS * GRAD_ACCUM}')
print(f'max_steps={MAX_STEPS} (save_steps={SAVE_STEPS})')

## 6. PatchDPOTrainer

In [ ]:
from unsloth import PatchDPOTrainer

PatchDPOTrainer()
print('PatchDPOTrainer applied.')

## 7. Load GRPO model + add LoRA

In [ ]:
import shutil
shutil.rmtree('/content/unsloth_compiled_cache', ignore_errors=True)

from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name     = HF_REPO_IN,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = False,
    fast_inference = False,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_R, lora_alpha = LORA_ALPHA, lora_dropout = LORA_DROPOUT, bias = "none",
    random_state = RANDOM_STATE, use_rslora = False, loftq_config = None,
    use_gradient_checkpointing = "unsloth",
)

if hasattr(FastVisionModel, 'for_training'):
    FastVisionModel.for_training(model)

print('Model + LoRA ready.')

## 8. Dataset

In [ ]:
from dpo_pipeline import strip_think_primer, build_dataset

strip_think_primer(tokenizer)
train_dataset = build_dataset(DPO_JSON, img_resolution=IMG_RESOLUTION, max_samples=MAX_SAMPLES)
print(train_dataset)

## 9. Trainer

In [ ]:
from dpo_pipeline import build_trainer

dpo_trainer = build_trainer(
    model, tokenizer, train_dataset, OUTPUT_DIR,
    learning_rate     = LEARNING_RATE,
    beta              = BETA,
    per_device_bs     = PER_DEVICE_BS,
    grad_accum        = GRAD_ACCUM,
    max_steps         = MAX_STEPS,
    save_steps        = SAVE_STEPS,
    max_prompt_len    = MAX_PROMPT_LEN,
    max_completion_len= MAX_COMPLETION_LEN,
    max_len           = MAX_LEN,
    adam_beta1        = ADAM_BETA1,
    adam_beta2        = ADAM_BETA2,
    weight_decay      = WEIGHT_DECAY,
    warmup_ratio      = WARMUP_RATIO,
    lr_scheduler      = LR_SCHEDULER,
    max_grad_norm     = MAX_GRAD_NORM,
    optim             = OPTIM,
    seed              = SEED,
)
print('Trainer ready.')

## 10. Train

In [ ]:
dpo_trainer.train()

## 11. Save metric log

In [ ]:
from dpo_pipeline import save_log

print('saved log ->', save_log(dpo_trainer, HF_REPO_OUT, BETA, LEARNING_RATE))

## 12. Save LoRA & push to Hub

In [ ]:
model.push_to_hub_merged(HF_REPO_OUT, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print('Pushed merged 16bit ->', HF_REPO_OUT)

model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
print('Pushed adapter ->', ADAPTER_REPO)